# P8b — WebQSP với GNN-RAG features thật (CWQ giữ heuristic)

**Vì sao có notebook này.** Lần chạy P8 bước 02–03 dùng scorer heuristic (tần suất shortest-path) cho pool ứng viên:
pool chỉ chứa đáp án ở 33% câu WebQSP, `random_k5` thắng `top5`, F1 mọi method < 7%. Số đó không so được với
Seminar 1 (GNN-RAG-all F1 40.0 / Rec 69.4 / cost 6.5; CGAR 40.7 / 68.8 / 5.1) và với bài báo GNN-RAG.
Notebook này chạy lại **cùng harness** nhưng WebQSP lấy ứng viên từ GNN ReaRev thật (spec `gnnrag:`), CWQ tạm giữ heuristic (spec `rog:`).

| Bước | Nội dung | Thời gian (A100) | Output |
|---|---|---|---|
| 00 | Mount Drive, deps, kiểm tra import | 3 phút | – |
| 01 | Tìm / dựng `gnn_features_train.pkl`, `gnn_features_webqsp_test.pkl`; kiểm tra trần coverage | 2–15 phút | `ckpt/gnn_features_*.pkl`, `results/g01_dataset_stats.json` |
| 02 | Baseline không học trên pool GNN | 15–30 phút | `results/g02_baselines` |
| 03 | Train CGAR trên WebQSP train (GNN) → eval WebQSP (GNN) + CWQ (heuristic) | 1–2 giờ | `results/g03_cgar` |
| 04 | Bảng chính 3 seed + sweep (tuỳ chọn, thêm PRISM) | 3–6 giờ | `results/g04_main` |
| 05 | Bảng đối chiếu heuristic ↔ GNN và các dòng dán vào P8 | 1 phút | `results/g05_compare.md` |

Mọi cell **bỏ qua nếu output đã có** (xoá thư mục `results/gXX_*` để chạy lại). Kết quả heuristic cũ ở `results/b02_*`, `b03_*` được giữ nguyên để đối chiếu ở bước 05.

## 00 · Thiết lập
**Mục tiêu:** mount Drive, cài deps, xác nhận `kgqa_bench` import được.
**Output kỳ vọng:** `cwd = .../colab/graph_rag`, GPU name, `kgqa_bench OK`.

In [ ]:
# ==== 00a. Mount Drive + đường dẫn (SỬA NẾU KHÁC) ====
from google.colab import drive; drive.mount('/content/drive')
DRIVE_BASE  = '/content/drive/MyDrive/ThS/Research/colab'          # = thư mục colab/ local
GRAPH_RAG   = f'{DRIVE_BASE}/graph_rag'
RESULTS     = f'{GRAPH_RAG}/results'
CKPT        = f'{GRAPH_RAG}/ckpt'
GNN_RAG_DIR = '/content/drive/MyDrive/ThS/Research/GNN-RAG'          # repo GNN-RAG (chỉ cần nếu phải dựng lại pkl)
RUN_DIR     = f'{DRIVE_BASE}/run'                                      # chứa checkpoint.zip (ReaRev_webqsp/CWQ.ckpt) và data zip tải từ README GNN-RAG
# Thư mục checkpoint CŨ của Seminar 1 (sẽ copy pkl từ đây nếu ckpt/ chưa có):
OLD_CKPT_DIRS = ['/content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/checkpoints',
                 '/content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/utils/checkpoints']

import os, sys, json, subprocess, time, shutil, glob
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True   # 2 runtime cùng import từ Drive → không ghi .pyc
for d in glob.glob(f'{GRAPH_RAG}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
import pickle
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
os.chdir(GRAPH_RAG); sys.path.insert(0, GRAPH_RAG)
print('cwd =', os.getcwd())
assert os.path.exists('kgqa_bench/cli.py'), 'kgqa_bench/ chưa được sync lên Drive'

In [ ]:
# ==== 00b. Deps + GPU + import check ====
!pip install -q datasets sentence-transformers transformers accelerate scikit-learn networkx tqdm matplotlib tabulate
import torch, pandas as pd
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
from kgqa_bench.data import load_spec, load_rog_hf, BenchDataset
from kgqa_bench.metrics import summarize_records, compare_records, matched_cost_table
print('kgqa_bench OK')

def run_bench(out, train=None, evals=(), methods=(), extra='', force=False):
    if os.path.exists(f'{out}/results.json') and not force:
        print(f'[skip] {out} đã có results.json'); return load_results(out)
    cmd = f'python -m kgqa_bench.cli run --device {DEVICE} --out {out} '
    if train: cmd += f'--train {train} '
    cmd += '--eval ' + ' '.join(evals) + ' --methods ' + ' '.join(methods) + ' ' + extra
    print('$', cmd); t0 = time.time()
    rc = subprocess.call(cmd, shell=True)
    print(f'exit={rc}  {(time.time()-t0)/60:.1f} min')
    return load_results(out)

def load_results(out):
    with open(f'{out}/results.json') as f: return json.load(f)

COLS = ['n','f1','cov','prec','hit_ctx','cost','n_tokens','latency_ms','ece','aurc','sel_acc@80','las_hit1']
def show(R, cols=COLS):
    for ds, by_m in R['summaries'].items():
        df = pd.DataFrame(by_m).T; df = df[[c for c in cols if c in df.columns]]
        print(f'\n=== {ds} ==='); display(df.round(3))

def show_cmp(R):
    rows = []
    for ds, pairs in R['comparisons'].items():
        for k, c in pairs.items():
            if not c or not c.get('n'): continue
            b = c['bootstrap']
            rows.append(dict(dataset=ds, pair=k, n=c['n'], delta=round(b['delta']*100,2),
                             ci=f"[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}]", p=round(b['p_value'],4),
                             mcnemar_p=round(c.get('mcnemar',{}).get('p_value', float('nan')),4), sig=c['significant_0.05']))
    display(pd.DataFrame(rows))

def gate(cond, msg): print(('✅ PASS  ' if cond else '❌ FAIL  ') + msg)
def records(out, ds_key, method):
    p = f'{out}/records_{ds_key.replace(":", "_")}_{method}.jsonl'
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

# Mốc Seminar 1 (WebQSP test, GNN features thật, A100) — chỉ để đối chiếu
SEM1 = {'gnnrag_all': dict(f1=.400, cov=.694, prec=.339, cost=6.5),
        'gnnrag_top10': dict(f1=.396, cost=5.1),
        'cgar': dict(f1=.407, cov=.688, prec=.353, cost=5.1),
        'ceiling': .73}

## 01 · Tìm hoặc dựng GNN-RAG features cho WebQSP
**Mục tiêu:** có 2 file `ckpt/gnn_features_train.pkl` (WebQSP train, từ `train.info` do P2 cell 18 tạo) và `ckpt/gnn_features_webqsp_test.pkl` (từ `test.info` GNN-RAG phát hành). Thứ tự tìm: `ckpt/` → thư mục checkpoint Seminar 1 (copy sang) → dựng lại từ repo GNN-RAG.
**Output kỳ vọng:** train ≈ 2.8k câu (qid `WebQTrn-…`), test 1628 câu (qid `WebQTest-…`); `answer_in_subgraph_rate` test ≈ 0.70–0.80 (Seminar 1: 0.73); `mean_n_candidates` = 20; `gnn_hit` trung bình ≈ 0.85–0.90 (ReaRev Hit@1 trong paper ≈ 0.80 Hit).
**Gate:** qid train ∩ test = ∅; trần coverage trong [0.65, 0.85]. Trần < 0.5 → pkl sai (tên thực thể chưa resolve) → dựng lại.
**Lập luận:** trần coverage của pool GNN (≈0.73) so với pool heuristic (hit_ctx 0.33) là lý do duy nhất số tuyệt đối sẽ tăng; CGAR không đổi.

In [ ]:
# ==== 01a. Tìm pkl có sẵn; thiếu thì copy từ Seminar 1 ====
WANT = {'train': ['gnn_features_train.pkl'],
        'test':  ['gnn_features_webqsp_test.pkl', 'gnn_features_test.pkl']}
FEAT = {}
for split, names in WANT.items():
    for n in names:
        p = f'{CKPT}/{n}'
        if os.path.exists(p): FEAT[split] = p; break
    if split not in FEAT:
        for d in OLD_CKPT_DIRS:
            for n in names:
                src = f'{d}/{n}'
                if os.path.exists(src):
                    dst = f'{CKPT}/{names[0]}'; print(f'copy {src} -> {dst}'); shutil.copy(src, dst); FEAT[split] = dst; break
            if split in FEAT: break
# C2 (11/2026): CWQ với features ReaRev_CWQ. BUILD_CWQ=False để chỉ làm WebQSP như trước.
BUILD_CWQ = True
FEAT_CWQ = {}
for split, n in (('train', 'gnn_features_cwq_train.pkl'), ('test', 'gnn_features_cwq_test.pkl')):
    if os.path.exists(f'{CKPT}/{n}'): FEAT_CWQ[split] = f'{CKPT}/{n}'
print('FEAT =', FEAT, '| FEAT_CWQ =', FEAT_CWQ)
if len(FEAT) < 2:
    print('THIẾU:', [s for s in WANT if s not in FEAT], '→ chạy cell 01b để dựng lại từ repo GNN-RAG')
    print('Các file trong ckpt/:'); print([f for f in os.listdir(CKPT) if f.endswith('.pkl')])

In [ ]:
# ==== 01b. (Chỉ khi 01a thiếu) Dựng pkl từ repo GNN-RAG — port P2 cell 30 / P7v3 cell 2 ====
# Cần: gnn/data/webqsp/{train,test}.json, entities.txt, relations.txt; llm/entities_names.json;
#      llm/results/gnn/RoG-webqsp/rearev-sbert/test.info  (test, GNN-RAG phát hành)
#      gnn/checkpoint/pretrain/train.info                   (train, P2 cell 18 tạo bằng cách eval ReaRev trên train.json)
def build_gnn_pkl(dataset, split, out_path):
    dd = 'webqsp' if dataset == 'webqsp' else 'CWQ'
    data_path = f'{GNN_RAG_DIR}/gnn/data/{dd}/{split}.json'
    info_path = (f'{GNN_RAG_DIR}/llm/results/gnn/RoG-{dataset}/rearev-sbert/test.info' if split == 'test'
                 else f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/' + ('train.info' if dataset == 'webqsp' else f'train_{dataset}.info'))
    ents, rels, names = f'{GNN_RAG_DIR}/gnn/data/{dd}/entities.txt', f'{GNN_RAG_DIR}/gnn/data/{dd}/relations.txt', f'{GNN_RAG_DIR}/llm/entities_names.json'
    for p in (data_path, info_path, ents, rels, names):
        if not os.path.exists(p): print('MISSING', p); return None
    id2mid = {i: l.strip() for i, l in enumerate(open(ents))}
    id2rel = {i: l.strip() for i, l in enumerate(open(rels))}
    mid2name = json.load(open(names))
    def ent(e):
        if isinstance(e, int): m = id2mid.get(e, str(e)); return mid2name.get(m, m)
        if isinstance(e, str) and e.startswith('m.'): return mid2name.get(e, e)
        return str(e)
    def rel(r): return id2rel.get(r, str(r)) if isinstance(r, int) else str(r)
    data = [json.loads(l) for l in open(data_path)]
    info = [json.loads(l) for l in open(info_path)]
    assert len(data) == len(info), f'{len(data)} data vs {len(info)} info — info không khớp split'
    feats = {}
    for d, g in zip(data, info):
        cand = g.get('cand', [])
        node_scores = {mid2name.get(m, m): s for m, s in cand}
        ans = [a.get('text', mid2name.get(a.get('kb_id', ''), str(a))) if isinstance(a, dict) else ent(a) for a in d.get('answers', [])]
        feats[d['id']] = dict(question=d['question'], node_scores=node_scores,
                              top_candidates=[mid2name.get(c[0], c[0]) for c in cand[:20]],
                              q_entities=[ent(e) for e in d.get('entities', [])], answers=ans,
                              subgraph_tuples=[[ent(h), rel(r), ent(t)] for h, r, t in d['subgraph']['tuples']],
                              gnn_f1=g.get('f1', 0), gnn_hit=g.get('hit', 0))
    pickle.dump(feats, open(out_path, 'wb')); print(f'saved {len(feats)} -> {out_path}'); return out_path

if 'test' not in FEAT:  FEAT['test']  = build_gnn_pkl('webqsp', 'test',  f'{CKPT}/gnn_features_webqsp_test.pkl')
if 'train' not in FEAT:
    if not os.path.exists(f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/train.info'):
        print('Chưa có train.info → chạy cell 01c (eval ReaRev trên train split, ~10 phút GPU) rồi chạy lại cell này')
    else:
        FEAT['train'] = build_gnn_pkl('webqsp', 'train', f'{CKPT}/gnn_features_train.pkl')
# C2: CWQ — test.info do GNN-RAG phát hành; train_cwq.info do cell 01c tạo bằng ReaRev_CWQ.ckpt
if BUILD_CWQ:
    if 'test' not in FEAT_CWQ:
        r = build_gnn_pkl('cwq', 'test', f'{CKPT}/gnn_features_cwq_test.pkl')
        if r: FEAT_CWQ['test'] = r
    if 'train' not in FEAT_CWQ:
        if os.path.exists(f'{GNN_RAG_DIR}/gnn/checkpoint/pretrain/train_cwq.info'):
            r = build_gnn_pkl('cwq', 'train', f'{CKPT}/gnn_features_cwq_train.pkl')
            if r: FEAT_CWQ['train'] = r
        else:
            print('CWQ: chưa có train_cwq.info → cell 01c sẽ eval ReaRev_CWQ trên CWQ train (~30 phút GPU, 27k câu)')
print('FEAT =', FEAT, '| FEAT_CWQ =', FEAT_CWQ)

In [ ]:
# ==== 01c. (Chỉ khi thiếu train.info) Eval ReaRev trên WebQSP train → train.info — port P2 cell 18 ====
# Cần checkpoint gnn/checkpoint/pretrain/ReaRev_webqsp.ckpt và repo GNN-RAG cài được (torch_geometric...).
import zipfile
g = f'{GNN_RAG_DIR}/gnn'
# Tham số ReaRev theo README GNN-RAG (nếu load ckpt báo lệch shape → chỉnh num_iter/num_ins/num_gnn cho khớp ckpt)
REAREV_ARGS = {'webqsp': dict(folder='webqsp', ckpt='ReaRev_webqsp.ckpt', args='--num_iter 3 --num_ins 2 --num_gnn 3 --num_epoch 200'),
               'cwq':    dict(folder='CWQ',    ckpt='ReaRev_CWQ.ckpt',    args='--num_iter 2 --num_ins 3 --num_gnn 3 --num_epoch 100')}
def ensure_gnnrag_repo():
    if not os.path.exists(f'{g}/main.py'):                     # chưa có repo → clone
        subprocess.call(f'git clone --depth 1 https://github.com/cmavro/GNN-RAG {GNN_RAG_DIR}', shell=True)
    if os.path.exists(f'{RUN_DIR}/checkpoint.zip') and not all(os.path.exists(f'{g}/checkpoint/pretrain/{v["ckpt"]}') for v in REAREV_ARGS.values()):
        with zipfile.ZipFile(f'{RUN_DIR}/checkpoint.zip') as z:   # README GNN-RAG "download released GNN models"
            for m in z.namelist():
                if m.startswith('checkpoint/') and not m.endswith('/') and '.DS_Store' not in m: z.extract(m, g)
        print('ckpt:', os.listdir(f'{g}/checkpoint/pretrain'))
    for folder in ('webqsp', 'CWQ'):
        if not os.path.exists(f'{g}/data/{folder}/train.json'):
            for zf in glob.glob(f'{RUN_DIR}/*.zip'):               # README GNN-RAG "download data" (data.zip có thể lồng trong zip khác)
                if 'checkpoint' in zf: continue
                with zipfile.ZipFile(zf) as z:
                    names = z.namelist()
                    inner = [n for n in names if n.endswith('data.zip')]
                    if inner:
                        z.extract(inner[0], RUN_DIR); zf = f'{RUN_DIR}/{inner[0]}'
                        with zipfile.ZipFile(zf) as z2: names = z2.namelist(); print('giải nén', zf); z2.extractall(g if names[0].startswith('data/') else f'{g}/data')
                    elif any(folder.lower() in n.lower() and n.endswith('train.json') for n in names):
                        print('giải nén data từ', zf); z.extractall(g if names[0].startswith('data/') else f'{g}/data')

def eval_rearev_on_train(dataset):
    """Chạy ReaRev (ckpt công bố) ở chế độ eval trên TRAIN split của `dataset` → gnn/checkpoint/pretrain/train[_cwq].info"""
    a = REAREV_ARGS[dataset]; d = f'{g}/data/{a["folder"]}'
    for p in (f'{g}/checkpoint/pretrain/{a["ckpt"]}', f'{d}/train.json', f'{d}/entities.txt'):
        assert os.path.exists(p), f'thiếu {p} — tải checkpoint.zip / data zip từ README GNN-RAG vào {RUN_DIR}'
    shutil.copy(f'{d}/test.json', f'{d}/test_backup.json'); shutil.copy(f'{d}/train.json', f'{d}/test.json')
    try:
        rc = subprocess.call(f'python main.py ReaRev --entity_dim 50 --batch_size 8 --eval_every 2 --data_folder data/{a["folder"]}/ '
                             f'--lm sbert {a["args"]} --relation_word_emb True --load_experiment {a["ckpt"]} --is_eval --name {dataset}_train_eval',
                             shell=True, cwd=g); print('exit', rc)
    finally:
        shutil.move(f'{d}/test_backup.json', f'{d}/test.json')
    infos = glob.glob(f'{g}/checkpoint/pretrain/*{dataset}_train_eval*.info') or glob.glob(f'{g}/checkpoint/pretrain/*test.info')
    if not infos: print('không thấy .info sau khi eval'); return None
    latest = max(infos, key=os.path.getmtime)
    dst = f'{g}/checkpoint/pretrain/' + ('train.info' if dataset == 'webqsp' else f'train_{dataset}.info')
    shutil.copy(latest, dst); print('->', latest, '=>', dst); return dst

if 'train' not in FEAT and not os.path.exists(f'{g}/checkpoint/pretrain/train.info'):
    ensure_gnnrag_repo()
    if eval_rearev_on_train('webqsp'): FEAT['train'] = build_gnn_pkl('webqsp', 'train', f'{CKPT}/gnn_features_train.pkl')
else:
    print('[skip] WebQSP: đã có train features hoặc train.info')
if BUILD_CWQ and 'train' not in FEAT_CWQ and not os.path.exists(f'{g}/checkpoint/pretrain/train_cwq.info'):
    ensure_gnnrag_repo()
    if eval_rearev_on_train('cwq'):
        r = build_gnn_pkl('cwq', 'train', f'{CKPT}/gnn_features_cwq_train.pkl')
        if r: FEAT_CWQ['train'] = r
    if 'test' not in FEAT_CWQ:
        r = build_gnn_pkl('cwq', 'test', f'{CKPT}/gnn_features_cwq_test.pkl')
        if r: FEAT_CWQ['test'] = r
elif BUILD_CWQ:
    print('[skip] CWQ: đã có features hoặc train_cwq.info')

In [ ]:
# ==== 01d. Kiểm tra pkl qua harness + trần coverage ====
assert 'train' in FEAT and 'test' in FEAT, 'thiếu features — xem 01a/01b/01c'
WEBQSP_TR = f"gnnrag:{FEAT['train']}:webqsp:train"
WEBQSP_TE = f"gnnrag:{FEAT['test']}:webqsp:test"
CWQ_TE    = f"gnnrag:{FEAT_CWQ['test']}:cwq:test" if FEAT_CWQ.get('test') else 'rog:rmanluo/RoG-cwq:test'   # C2: GNN nếu đã dựng
DATA_CACHE = f'{RESULTS}/data_cache'

tr, te = load_spec(WEBQSP_TR), load_spec(WEBQSP_TE)
cwq = load_spec(CWQ_TE) if CWQ_TE.startswith('gnnrag:') else load_rog_hf('rmanluo/RoG-cwq', 'test', cache_dir=DATA_CACHE)
CWQ_TAG = '(gnn)' if CWQ_TE.startswith('gnnrag:') else '(heur)'
stats = {d.key + ('(gnn)' if d is not cwq else CWQ_TAG): d.stats() for d in (tr, te, cwq)}
df = pd.DataFrame(stats).T; display(df.round(3))
import numpy as np
gnn_hit = np.mean([q.meta.get('gnn_hit', 0) for q in te]); print('mean gnn_hit (ReaRev Hit, test) =', round(gnn_hit, 3))
print('qid ví dụ:', tr.qids()[:2], te.qids()[:2])
gate(not (set(tr.qids()) & set(te.qids())), 'train ∩ test = ∅')
gate(0.70 <= gnn_hit <= 0.80, f'mean gnn_hit = {gnn_hit:.3f} ≈ ReaRev Hit 76.4 trong paper GNN-RAG → pkl đúng')
ceil = stats['webqsp:test(gnn)']['answer_in_subgraph_rate']
gate(0.80 <= ceil <= 0.95, f'trần SUBGRAPH WebQSP(gnn) = {ceil:.3f} trong [0.80, 0.95] (subgraph GNN-RAG = subgraph RoG; đo 11/2026: 0.887)')
# Con số thực sự giới hạn recall: đáp án có nằm trong POOL ỨNG VIÊN ReaRev không (Seminar 1 gọi là ~73%)
pool_hit = {k: float(np.mean([any(a in set(q.top_candidates) for a in q.answers) for q in d if q.answers])) for k, d in (('train', tr), ('test', te), ('cwq' + CWQ_TAG, cwq))}
print('answer-in-candidate-pool:', {k: round(v, 3) for k, v in pool_hit.items()})
gate(0.65 <= pool_hit['test'] <= 0.85, f"đáp án trong pool ứng viên GNN (test) = {pool_hit['test']:.3f} (Seminar 1: ≈0.73); heuristic chỉ ≈0.33")
gate(stats['webqsp:test(gnn)']['mean_n_candidates'] >= 3, f"ReaRev xuất trung bình {stats['webqsp:test(gnn)']['mean_n_candidates']:.1f} ứng viên/câu (chỉ ứng viên vượt ngưỡng; 11/2026: 5.6) → pool 10 ≈ pool 20, K của CGAR gộp một mức")
if CWQ_TE.startswith('gnnrag:'):
    cwq_hit = np.mean([q.meta.get('gnn_hit', 0) for q in cwq]); print('mean gnn_hit (ReaRev Hit, CWQ test) =', round(cwq_hit, 3))
    gate(0.45 <= cwq_hit <= 0.62, f'CWQ: mean gnn_hit = {cwq_hit:.3f} ≈ ReaRev Hit 52.9 trong paper GNN-RAG → pkl CWQ đúng')
    gate(pool_hit['cwq(gnn)'] > 0.45, f"CWQ: đáp án trong pool ReaRev = {pool_hit['cwq(gnn)']:.3f} (heuristic ≈0.33)")
stats['pool_hit'] = pool_hit
json.dump(stats, open(f'{RESULTS}/g01_dataset_stats.json', 'w'), indent=2)
print('WEBQSP_TR =', WEBQSP_TR); print('WEBQSP_TE =', WEBQSP_TE); print('CWQ_TE    =', CWQ_TE)

## 01e · (C1, tuỳ chọn) Pool ứng viên rộng hơn từ ReaRev
ReaRev chỉ xuất ~5.6 ứng viên/câu (trên ngưỡng), nên pool 10 ≈ pool 20 và mọi cơ chế chọn ứng viên (K-level) vô hiệu. Muốn pool 20 thật phải sửa chỗ GNN-RAG cắt `cand` khi ghi `.info`, rồi chạy lại 01c/01b. Cell này chỉ **tìm và in** chỗ đó trong repo để sửa tay; không tự sửa code của GNN-RAG.


In [ ]:
g = f'{GNN_RAG_DIR}/gnn'
if os.path.exists(g):
    print(subprocess.run(f"grep -rn \"'cand'\\|\\\"cand\\\"\\|cand_list\\|top_k\\|topk\" {g}/*.py {g}/models {g}/train_model.py 2>/dev/null | head -40", shell=True, capture_output=True, text=True).stdout)
    print('→ Tìm dòng tạo danh sách cand (thường lọc theo xác suất > ngưỡng hoặc top-k). Nới ngưỡng / tăng k lên 20, chạy lại eval test + 01c, rồi build lại pkl với hậu tố _pool20 và đổi WEBQSP_TE/TR.')
else:
    print('[skip] chưa clone repo GNN-RAG (01c sẽ clone khi cần)')


## 02 · Baseline không học trên pool GNN
**Mục tiêu:** đường cơ sở cost–quality với ứng viên GNN thật. Lần heuristic FAIL gate `random_k5 < top5`; với GNN phải PASS.
**Output kỳ vọng (WebQSP, so với Seminar 1):** `gnnrag_all` F1 ≈ 0.40 / cov ≈ 0.69 / cost ≈ 6.5; `gnnrag_top10` F1 ≈ 0.40 / cost ≈ 5.1; `random_k5` thấp hơn `top5` rõ rệt. Cost `all` chỉ ≈ 6.5 vì ứng viên GNN có ít đường nối hơn pool heuristic (26.9).
**Pool:** mọi baseline dùng 20 ứng viên (mặc định `--baseline-candidates 20`, bằng pool CGAR với tới). Seminar 1 tính "all" trên 10 ứng viên nên cov/cost của `gnnrag_all` ở đây sẽ cao hơn một chút; dòng `gnnrag_all10` chạy thêm để đối chiếu đúng mốc cũ. `sbert_topK` xếp đường theo cosine(câu hỏi, đường) bằng chính embedder của CGAR, không học.
**Gate:** random_k5 < top5 (CI không chồng); Rec giảm đơn điệu theo K; |F1(all, 10 ứng viên) − 0.40| ≤ 0.03.
**Lập luận:** nếu gate 1 PASS thì ranking GNN có tín hiệu và nhãn `gnnrag_topK` là đúng nghĩa; với CWQ (heuristic) gate này vẫn FAIL và các dòng CWQ phải ghi "heuristic-topK".

In [ ]:
G02 = f'{RESULTS}/g02_baselines'
R02 = run_bench(G02, evals=[WEBQSP_TE, CWQ_TE],
                methods=['gnnrag_all','gnnrag_top10','gnnrag_top5','gnnrag_top3','sbert_top10','sbert_top5','graphrag_fi','random_k5'],
                extra='--compare-ref gnnrag_top5 --quiet')
show(R02)
# Mốc Seminar 1 "GNN-RAG-all" = 10 ứng viên → chạy riêng để đối chiếu đúng
R02b = run_bench(f'{RESULTS}/g02_baselines_pool10', evals=[WEBQSP_TE], methods=['gnnrag_all'], extra='--baseline-candidates 10 --quiet')
s10 = R02b['summaries']['webqsp:test']['gnnrag_all']; print('gnnrag_all (10 ứng viên, như Seminar 1): F1 %.3f cov %.3f cost %.2f' % (s10['f1'], s10['cov'], s10['cost']))
s = R02['summaries']['webqsp:test']
gate(s['gnnrag_all']['cov'] > s['gnnrag_top10']['cov'] > s['gnnrag_top5']['cov'] > s['gnnrag_top3']['cov'], 'Rec giảm đơn điệu theo K')
gate(s['random_k5']['hit_ctx_ci_high'] < s['gnnrag_top5']['hit_ctx_ci_low'], 'random_k5 < top5 (CI không chồng) — ranking GNN có tín hiệu')
gate(abs(s10['f1'] - SEM1['gnnrag_all']['f1']) <= 0.03, f"F1 gnnrag_all(10) = {s10['f1']:.3f} ≈ Seminar 1 (0.400 ± 0.03)")
gate(abs(s10['cov'] - SEM1['gnnrag_all']['cov']) <= 0.05, f"cov gnnrag_all(10) = {s10['cov']:.3f} ≈ Seminar 1 (0.694)")
TOPK_AT_COST5 = s['gnnrag_top5']['f1']; TOPK_AT_COST10 = s['gnnrag_top10']['f1']
SBERT_AT_COST5 = s['sbert_top5']['f1']; SBERT_AT_COST10 = s['sbert_top10']['f1']
print('mốc F1 top-5 (cost %.1f) = %.2f | top-10 (cost %.1f) = %.2f' % (s['gnnrag_top5']['cost'], TOPK_AT_COST5*100, s['gnnrag_top10']['cost'], TOPK_AT_COST10*100))
print('mốc F1 sbert_top5 = %.2f | sbert_top10 = %.2f' % (SBERT_AT_COST5*100, SBERT_AT_COST10*100))
sc = R02['summaries']['cwq:test']
gate(sc['random_k5']['hit_ctx_ci_high'] < sc['gnnrag_top5']['hit_ctx_ci_low'], 'CWQ(heuristic): random_k5 < top5 — kỳ vọng FAIL, ghi nhãn heuristic-topK trong report')

## 03 · Huấn luyện CGAR trên WebQSP (GNN) và đánh giá chéo
**Mục tiêu:** tái lập kết quả Seminar 1 trong harness mới (checkpoint chọn trên dev tách từ train, không phải test), rồi đo transfer sang CWQ heuristic.
**Cấu hình:** `--cgar-max-paths 5` để cost ≈ 5 như Seminar 1 (so với `top10` cost 5.1); `--dev-split 0.1`; S1 10 epoch, S2 300 bước.
**Output kỳ vọng:** WebQSP F1 ≈ 0.40–0.42, cov ≈ 0.69, cost ≈ 5; `rel_loss` < 0.5; `advantage_nonzero_frac` thấp (S2 gần như không học, như lần trước); ECE/AURC tốt hơn top-10. CWQ: thấp (pool heuristic) nhưng vẫn ≥ top-10 heuristic.
**Gate:** F1 CGAR ≥ F1 top-10 (cùng cost ≈ 5) − 0.005; F1 CGAR ≥ sbert_top5 + 1 pp; |F1 − 0.407| ≤ 0.03; cost ≤ 6. Mục *Adaptivity* trong `report.md` phải không phải "constant".
**Lập luận:** CGAR chỉ chọn đường trong pool; khi pool GNN tốt, CGAR phải bằng/hơn top-10 ở cùng cost với confidence dùng được (AURC thấp hơn). Nếu F1 thấp hơn top-10 rõ → relevance head overfit, xem history epoch.

In [ ]:
G03 = f'{RESULTS}/g03_cgar'
R03 = run_bench(G03, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['cgar'],
                extra='--dev-split 0.1 --cgar-max-paths 5 --cgar-s1-epochs 10 --cgar-s2-steps 300 --compare-ref cgar --quiet')
show(R03)
log = R03['train_logs']['cgar@seed42']
h1 = pd.DataFrame(log['stage1']['history']); display(h1[['epoch','loss','rel_loss','eval_f1','eval_cost']].round(4))
print('S2 advantage_nonzero_frac =', round(log['stage2'].get('advantage_nonzero_frac', float('nan')), 3))
print('S2 eval_f1 history:', [(r['step'], round(r['eval_f1']*100, 2)) for r in log['stage2'].get('history', [])])
sw = R03['summaries']['webqsp:test']['cgar']
gate(h1['rel_loss'].iloc[-1] < 0.6, 'rel_loss < 0.6')
gate(sw['f1'] >= TOPK_AT_COST10 - 0.005, f"F1 CGAR {sw['f1']:.3f} ≥ top-10 {TOPK_AT_COST10:.3f} (cùng cost ≈ 5)")
gate(abs(sw['f1'] - SEM1['cgar']['f1']) <= 0.03, f"F1 CGAR {sw['f1']:.3f} ≈ Seminar 1 0.407")
gate(sw['cost'] <= 6.0, f"cost CGAR = {sw['cost']:.2f} ≤ 6")
gate(sw['aurc'] < s['gnnrag_top10']['aurc'], f"AURC CGAR {sw['aurc']:.3f} < top-10 {s['gnnrag_top10']['aurc']:.3f}")
_ref = SBERT_AT_COST5 if sw['cost'] <= 7.5 else SBERT_AT_COST10
gate(sw['f1'] >= _ref + 0.01, f"F1 CGAR {sw['f1']:.3f} ≥ sbert_topK cùng cost {_ref:.3f} + 1 pp (gain từ controller, không chỉ embedding)")
# chosen_k / confidence: cơ chế adaptive có hoạt động không?
from collections import Counter
rc = records(G03, 'webqsp:test', 'cgar')
print('chosen_k:', Counter(r.get('chosen_k') for r in rc)); print('cost histogram:', Counter(round(r['cost']) for r in rc))
rep = open(f'{G03}/report.md').read(); print(rep.split('## Adaptivity')[-1][:500] if '## Adaptivity' in rep else '')
gate(len(Counter(r.get('chosen_k') for r in rc)) > 1 or len(Counter(round(r['cost']) for r in rc)) > 1, 'K hoặc số đường thay đổi theo câu (adaptive thật)')
CGAR_CKPT = f'{G03}/ckpt/cgar_s1_seed42.pt'
if os.path.exists(f'{G03}/ckpt/cgar_s2_seed42.pt') and log['stage2'].get('best_eval_f1', 0) > log['stage1']['best_eval_f1'] + 0.002:
    CGAR_CKPT = f'{G03}/ckpt/cgar_s2_seed42.pt'
print('CGAR_CKPT =', CGAR_CKPT)

## 04 · Bảng chính 3 seed + sweep (tuỳ chọn)
**Mục tiêu:** bảng để trích vào báo cáo: mọi method cùng pool GNN, mean ± std theo seed, Δ với CI và p so với `gnnrag_all`, đường Pareto.
**Output kỳ vọng:** std F1 theo seed < 1 pp; CGAR = GNN-RAG-all về F1 (CI chứa 0) với cost thấp hơn ≥ 20%; `cgar_beta_sweep` cho đường cost–F1.
**Gate:** std < 0.01; (Δf1 ≥ 0 hoặc không ý nghĩa) và cost_cgar ≤ 0.8 × cost_all.
Đặt `RUN_PRISM = True` để thêm PRISM (teacher oracle, nhãn `oracle_heuristic`) vào cùng bảng.

In [ ]:
RUN_PRISM = False
G04 = f'{RESULTS}/g04_main'
methods = ['gnnrag_all','gnnrag_top10','gnnrag_top5','sbert_top10','sbert_top5','graphrag_fi','cgar'] + (['prism'] if RUN_PRISM else [])
R04 = run_bench(G04, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=methods,
                extra='--seeds 42 1337 2024 --sweep --dev-split 0.1 --cgar-max-paths 5 --compare-ref gnnrag_all --n-bootstrap 10000 --quiet')
show(R04); show_cmp(R04)
for ds, by_m in R04['seeds'].items():
    for m in [x for x in ('cgar','prism') if x in by_m]:
        sd = by_m[m]['f1']; print(ds, m, 'F1 = %.2f ± %.2f' % (sd['mean']*100, sd['std']*100))
        gate(sd['std'] < 0.01, f'{ds} {m}: std F1 theo seed < 1 pp')
s4 = R04['summaries']['webqsp:test']; c4 = R04['comparisons']['webqsp:test'].get('cgar_vs_gnnrag_all:f1', {})
gate(bool(c4) and (c4['bootstrap']['delta'] >= 0 or not c4['significant_0.05']) and s4['cgar']['cost'] <= 0.8 * s4['gnnrag_all']['cost'],
     'CGAR ≈ GNN-RAG-all về F1 với cost thấp hơn ≥ 20%')
print('matched-cost vs cgar:', matched_cost_table(s4, 'cgar', tol=0.75))
display(pd.DataFrame(R04['train_logs'].get('cgar_beta_sweep', [])))
from IPython.display import Image; display(Image(f'{G04}/pareto.png'))

## 05 · Đối chiếu heuristic ↔ GNN và các dòng dán vào P8
**Mục tiêu:** một bảng cho báo cáo giải thích vì sao số lần trước thấp: cùng method, cùng harness, chỉ đổi pool ứng viên. Đồng thời in 3 dòng spec để P8 bước 04–11 chạy tiếp trên features thật.
**Output kỳ vọng:** cột `gnn` cao hơn `heur` 5–10 lần ở F1/cov; cost `all` giảm (26.9 → ≈6.5); CGAR giữ thứ hạng trên cả hai pool.

In [ ]:
rows = []
B02, B03 = f'{RESULTS}/b02_baselines', f'{RESULTS}/b03_cgar'
srcs = {'heur': {}, 'gnn': {}}
if os.path.exists(f'{B02}/results.json'): srcs['heur'].update(load_results(B02)['summaries'].get('webqsp:test', {}))
if os.path.exists(f'{B03}/results.json'): srcs['heur'].update(load_results(B03)['summaries'].get('webqsp:test', {}))
srcs['gnn'].update(R02['summaries']['webqsp:test']); srcs['gnn'].update(R03['summaries']['webqsp:test'])
for m in ['gnnrag_all','gnnrag_top10','gnnrag_top5','sbert_top10','sbert_top5','graphrag_fi','random_k5','cgar']:
    for pool in ('heur','gnn'):
        if m in srcs[pool]:
            x = srcs[pool][m]; rows.append(dict(method=m, pool=pool, f1=x['f1'], cov=x['cov'], hit_ctx=x['hit_ctx'], cost=x['cost'], aurc=x.get('aurc')))
cmp = pd.DataFrame(rows).pivot_table(index='method', columns='pool', values=['f1','cov','hit_ctx','cost','aurc']).round(3)
display(cmp)
sem1 = pd.DataFrame(SEM1).T.drop('ceiling', errors='ignore'); print('Seminar 1 (WebQSP, GNN):'); display(sem1)
with open(f'{RESULTS}/g05_compare.md', 'w') as f:
    f.write('# WebQSP test: heuristic pool vs GNN-RAG pool (cùng harness)\n\n'); f.write(cmp.to_markdown()); f.write('\n\n## Seminar 1\n\n'); f.write(sem1.to_markdown())
print('saved', f'{RESULTS}/g05_compare.md')
print('\nDán vào cell 00d của P8 để các bước 04–11 chạy trên features thật:')
print(f"WEBQSP_TR = '{WEBQSP_TR}'"); print(f"WEBQSP_TE = '{WEBQSP_TE}'"); print(f"CWQ_TE    = '{CWQ_TE}'")
print(f"CGAR_CKPT = '{CGAR_CKPT}'")